# Metamodel case generation

Workflow:

1. Load **goals** from the case JSON → `simulations.goals_file`.
2. Generate **Latin Hypercube** cases per goal (`hs`, `tp`, `dir`, `spr`, `wl`)
   and keep only plausible sea states (steepness `hs / L0 <= max_steepness`), so
   each goal keeps fewer cases than `num_samples_per_goal`.
3. Build SnapWave case directories from `templates-metamodel`.
4. Run cases (optional) and postprocess to per-goal NetCDF files.

Paths and bounds come from the case JSON → `simulations`.

In [ ]:
from bluemath_tk.wrappers.snapwave import SnapWaveMetaModelWrapper
from utils.config import PATHS, SIMULATIONS, load_goals, resolve
from utils import simulation as sim

## Configuration

In [ ]:
templates_dir = sim.templates_dir("metamodel")
cases_dir = resolve(SIMULATIONS["metamodel_cases"])
aoi_file = resolve(PATHS["aoi_file"])
mesh_file = resolve(PATHS["mesh_file"])
goals_file = resolve(SIMULATIONS["goals_file"])

print(f"goals     : {goals_file}")
print(f"templates : {templates_dir}")
print(f"cases     : {cases_dir}")
print(f"mesh      : {mesh_file}")
print(f"LHS       : {SIMULATIONS['num_samples_per_goal']} samples per goal")
print(f"bounds    : {SIMULATIONS['bounds']} (dir from each goal sector)")
print(f"steepness : hs / L0 <= {SIMULATIONS['max_steepness']}")

## LHS cases

In [ ]:
raw_goals = load_goals()

cases_df = sim.generate_cases(raw_goals)
cases_df["goal"] = cases_df["goal"].astype(int)
print(f"plausible cases per goal: {cases_df.groupby('goal').size().to_dict()}")
cases_df.head()

In [ ]:
import numpy as np

# Add a new column called save_map with 5% True
cases_df["save_map"] = np.random.choice(
    [True, False], size=len(cases_df), p=[0.05, 0.95]
)
cases_df.head(20)

In [ ]:
cases_df.groupby(["goal", "save_map"])[["tp"]].count()

## SnapWave wrapper

Create the wrapper, then run `plot_configuration()` for a **pre-run diagnostic**:
left side = domain map + direction-range bars per goal; right side = text summary.
Review this before `build_cases()`.

In [ ]:
# Case geometry: goals = boundary nodes, AoI enclosure, labelled output sites
setup = sim.SnapWaveCaseSetup.from_case(raw_goals)

wrapper = SnapWaveMetaModelWrapper(
    templates_dir=str(templates_dir),
    metamodel_parameters=cases_df.to_dict(orient="list"),
    fixed_parameters={"gridfile": str(mesh_file)},
    output_dir=str(cases_dir),
    **setup.metamodel_kwargs(),
)
wrapper

In [ ]:
fig = setup.plot_metamodel_configuration(cases_df)
# fig.savefig("metamodel_setup_check.png", dpi=150, bbox_inches="tight")

## Build, run, postprocess

In [ ]:
wrapper.build_cases(
    cases_name_format=lambda ctx: f"{ctx.get('goal'):02}/{ctx.get('case_num'):04}"
)

In [ ]:
# On the GeoOcean cluster: one SLURM array task per case. Writes case_dirs.txt
# and snapwave_array.sh (module load snapwave, OMP threads, scratch cleanup)
# into the cases folder, then submits it with sbatch.
# wrapper.run_cases_slurm(max_parallel=200)
# Or locally, with snapwave on the PATH:
wrapper.run_cases_in_background(launcher="default", num_workers=2)

In [ ]:
cases_status = wrapper.monitor_cases(value_counts="simple")
cases_status

## Inspect finished cases

Post-run diagnostic (reads directly from the case folder):

- top row: full-width domain `tripcolor` map (or overview fallback)
- bottom 2×2: `hs`, `tp`, `dir`, `spr` at output sites
- title/subtitle: case id + boundary forcing summary

Customize colormaps via `CasePlotStyle(cmaps={"hs": "viridis", "tp": "plasma", "dir": "twilight_shifted", "spr": "cividis"})`.

In [ ]:
from bluemath_tk.wrappers.snapwave.snapwave_plots import CasePlotStyle
from utils.viz import plot_metamodel_case

style = CasePlotStyle(
    cmaps={"hs": "viridis", "tp": "plasma", "dir": "twilight_shifted", "spr": "cividis"}
)
plot_metamodel_case(cases_dir / "10" / "4800", style=style, show_satellite=True);

In [ ]:
wrapper.postprocess_cases()